In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    precision_recall_curve,
    auc
)

from xgboost import XGBClassifier

In [19]:
df = pd.read_csv("american_bankruptcy.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (78682, 21)


,company_name,status_label,year,Current assets,Cost of goods sold,Depreciation and amortization,EBITDA,Inventory,Net Income,Total Receivables,...,Net sales,Total assets,Total Long-term debt,EBIT,Gross Profit,Total Current Liabilities,Retained Earnings,Total Revenue,Total Liabilities,Total Operating Expenses
0,C_1,alive,1999,511.267,833.107,18.373,89.031,336.018,35.163,128.348,...,1024.333,740.998,180.447,70.658,191.226,163.816,201.026,1024.333,401.483,935.302
1,C_1,alive,2000,485.856,713.811,18.577,64.367,320.590,18.531,115.187,...,874.255,701.854,179.987,45.790,160.444,125.392,204.065,874.255,361.642,809.888
2,C_1,alive,2001,436.656,526.477,22.496,27.207,286.588,-58.939,77.528,...,638.721,710.199,217.699,4.711,112.244,150.464,139.603,638.721,399.964,611.514
3,C_1,alive,2002,396.412,496.747,27.172,30.745,259.954,-12.410,66.322,...,606.337,686.621,164.658,3.573,109.590,203.575,124.106,606.337,391.633,575.592
4,C_1,alive,2003,432.204,523.302,26.680,47.491,247.245,3.504,104.661,...,651.958,709.292,248.666,20.811,128.656,131.261,131.884,651.958,407.608,604.467


In [20]:
if 'company_name' in df.columns:
    df.drop(columns=['company_name'], inplace=True)

In [21]:
df.fillna(df.median(numeric_only=True), inplace=True)

,status_label,year,Current assets,Cost of goods sold,Depreciation and amortization,EBITDA,Inventory,Net Income,Total Receivables,Market value,Net sales,Total assets,Total Long-term debt,EBIT,Gross Profit,Total Current Liabilities,Retained Earnings,Total Revenue,Total Liabilities,Total Operating Expenses
0,alive,1999,511.267,833.107,18.373,89.031,336.018,35.163,128.348,372.7519,1024.333,740.998,180.447,70.658,191.226,163.816,201.026,1024.333,401.483,935.302
1,alive,2000,485.856,713.811,18.577,64.367,320.590,18.531,115.187,377.1180,874.255,701.854,179.987,45.790,160.444,125.392,204.065,874.255,361.642,809.888
2,alive,2001,436.656,526.477,22.496,27.207,286.588,-58.939,77.528,364.5928,638.721,710.199,217.699,4.711,112.244,150.464,139.603,638.721,399.964,611.514
3,alive,2002,396.412,496.747,27.172,30.745,259.954,-12.410,66.322,143.3295,606.337,686.621,164.658,3.573,109.590,203.575,124.106,606.337,391.633,575.592
4,alive,2003,432.204,523.302,26.680,47.491,247.245,3.504,104.661,308.9071,651.958,709.292,248.666,20.811,128.656,131.261,131.884,651.958,407.608,604.467
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78677,alive,2014,233.211,43.338,14.094,45.615,3.376,25.261,22.846,756.4827,104.223,1099.101,184.666,31.521,60.885,28.197,28.095,104.223,225.887,58.608
78678,alive,2015,105.559,59.184,42.592,202.133,2.288,129.688,54.611,527.5750,291.153,1865.926,770.103,159.541,231.969,88.128,157.783,291.153,880.327,89.020
78679,alive,2016,63.971,69.074,65.057,79.051,2.581,-1.442,42.467,578.8868,169.858,1746.235,683.985,13.994,100.784,85.765,156.341,169.858,770.233,90.807
78680,alive,2017,135.207,66.527,65.330,69.171,2.013,-20.401,27.217,412.6241,161.884,1736.110,694.035,3.841,95.357,82.010,135.941,161.884,776.697,92.713


In [22]:
df.isnull().sum()

status_label                     0
year                             0
Current assets                   0
Cost of goods sold               0
Depreciation and amortization    0
EBITDA                           0
Inventory                        0
Net Income                       0
Total Receivables                0
Market value                     0
Net sales                        0
Total assets                     0
Total Long-term debt             0
EBIT                             0
Gross Profit                     0
Total Current Liabilities        0
Retained Earnings                0
Total Revenue                    0
Total Liabilities                0
Total Operating Expenses         0
dtype: int64

In [23]:
le = LabelEncoder()

df['status_label'] = le.fit_transform(
    df['status_label']
)

print("\nClass Mapping:")

for i, cls in enumerate(le.classes_):
    print(f"{cls} -> {i}")


Class Mapping:
alive -> 0
failed -> 1


In [24]:
df['debt_ratio'] = (
    df['Total Liabilities']
    / (df['Total assets'] + 1)
)

df['profit_margin'] = (
    df['Net Income']
    / (df['Total Revenue'] + 1)
)

df['expense_ratio'] = (
    df['Total Operating Expenses']
    / (df['Total Revenue'] + 1)
)

df['asset_turnover'] = (
    df['Net sales']
    / (df['Total assets'] + 1)
)

df['debt_earnings_ratio'] = (
    df['Total Liabilities']
    / (df['Retained Earnings'] + 1)
)

In [25]:
df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

df.fillna(0, inplace=True)

,status_label,year,Current assets,Cost of goods sold,Depreciation and amortization,EBITDA,Inventory,Net Income,Total Receivables,Market value,...,Total Current Liabilities,Retained Earnings,Total Revenue,Total Liabilities,Total Operating Expenses,debt_ratio,profit_margin,expense_ratio,asset_turnover,debt_earnings_ratio
0,0,1999,511.267,833.107,18.373,89.031,336.018,35.163,128.348,372.7519,...,163.816,201.026,1024.333,401.483,935.302,0.541084,0.034294,0.912193,1.380506,1.987284
1,0,2000,485.856,713.811,18.577,64.367,320.590,18.531,115.187,377.1180,...,125.392,204.065,874.255,361.642,809.888,0.514534,0.021172,0.925317,1.243864,1.763548
2,0,2001,436.656,526.477,22.496,27.207,286.588,-58.939,77.528,364.5928,...,150.464,139.603,638.721,399.964,611.514,0.562380,-0.092132,0.955907,0.898090,2.844633
3,0,2002,396.412,496.747,27.172,30.745,259.954,-12.410,66.322,143.3295,...,203.575,124.106,606.337,391.633,575.592,0.569548,-0.020433,0.947731,0.881790,3.130409
4,0,2003,432.204,523.302,26.680,47.491,247.245,3.504,104.661,308.9071,...,131.261,131.884,651.958,407.608,604.467,0.573860,0.005366,0.925736,0.917873,3.067397
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78677,0,2014,233.211,43.338,14.094,45.615,3.376,25.261,22.846,756.4827,...,28.197,28.095,104.223,225.887,58.608,0.205333,0.240071,0.556988,0.094739,7.763774
78678,0,2015,105.559,59.184,42.592,202.133,2.288,129.688,54.611,527.5750,...,88.128,157.783,291.153,880.327,89.020,0.471538,0.443904,0.304703,0.155953,5.544214
78679,0,2016,63.971,69.074,65.057,79.051,2.581,-1.442,42.467,578.8868,...,85.765,156.341,169.858,770.233,90.807,0.440830,-0.008440,0.531476,0.097215,4.895310
78680,0,2017,135.207,66.527,65.330,69.171,2.013,-20.401,27.217,412.6241,...,82.010,135.941,161.884,776.697,92.713,0.447120,-0.125249,0.569196,0.093192,5.671764


In [26]:
X = df.drop(columns=['status_label'])

y = df['status_label']

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Shape: (62945, 24)
Testing Shape: (15737, 24)


In [28]:
negative_class = (y_train == 0).sum()

positive_class = (y_train == 1).sum()

scale_pos_weight = (
    negative_class
    / positive_class
)

print(
    "Scale Pos Weight:",
    scale_pos_weight
)

Scale Pos Weight: 14.073036398467433


In [29]:
model = XGBClassifier(

    n_estimators=300,

    max_depth=6,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    scale_pos_weight=scale_pos_weight,

    objective='binary:logistic',

    eval_metric='logloss',

    random_state=42
)

In [30]:
model.fit(
    X_train,
    y_train
)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [31]:
y_probs = model.predict_proba(
    X_test
)[:, 1]

In [32]:
threshold = 0.5

y_pred = (
    y_probs >= threshold
).astype(int)

In [33]:
print(
    "\nCONFUSION MATRIX"
)

cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)


CONFUSION MATRIX
[[12187  2506]
 [  301   743]]


In [34]:
print(
    "\nCLASSIFICATION REPORT"
)

print(
    classification_report(
        y_test,
        y_pred
    )
)


CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.98      0.83      0.90     14693
           1       0.23      0.71      0.35      1044

    accuracy                           0.82     15737
   macro avg       0.60      0.77      0.62     15737
weighted avg       0.93      0.82      0.86     15737



In [35]:
precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

roc_auc = roc_auc_score(
    y_test,
    y_probs
)

precision_curve, recall_curve, _ = (
    precision_recall_curve(
        y_test,
        y_probs
    )
)

pr_auc = auc(
    recall_curve,
    precision_curve
)

print("\nMETRICS")

print(
    f"Precision : {precision:.4f}"
)

print(
    f"Recall    : {recall:.4f}"
)

print(
    f"F1 Score  : {f1:.4f}"
)

print(
    f"ROC-AUC   : {roc_auc:.4f}"
)

print(
    f"PR-AUC    : {pr_auc:.4f}"
)


METRICS
Precision : 0.2287
Recall    : 0.7117
F1 Score  : 0.3461
ROC-AUC   : 0.8600
PR-AUC    : 0.3857


In [38]:
print(X.columns.tolist())

['year', 'Current assets', 'Cost of goods sold', 'Depreciation and amortization', 'EBITDA', 'Inventory', 'Net Income', 'Total Receivables', 'Market value', 'Net sales', 'Total assets', 'Total Long-term debt', 'EBIT', 'Gross Profit', 'Total Current Liabilities', 'Retained Earnings', 'Total Revenue', 'Total Liabilities', 'Total Operating Expenses', 'debt_ratio', 'profit_margin', 'expense_ratio', 'asset_turnover', 'debt_earnings_ratio']


In [37]:
import joblib

joblib.dump(
    model,
    "bankruptcy_xgb.pkl"
)

joblib.dump(
    le,
    "label_encoder.pkl"
)

print(
    "Model Saved Successfully"
)

Model Saved Successfully
